# soulclip on Kaggle — 2x T4, 30 guaranteed GPU hours a week

Kaggle beats Colab for this job on two counts:

| | Kaggle | Colab free |
|---|---|---|
| Weekly GPU | **30 h guaranteed** | 15-30 h, varies with demand |
| GPU | **2x T4 (32 GB total)** | 1x T4 |
| Session | 9-12 h | 12 h |
| Background execution | **yes** | no (Pro+ only) |
| Download speed | ~1-2 GB/s | ~500 Mb/s |

The two T4s matter: this notebook runs **two workers in parallel**, one per
GPU, each on half the scenes, into a shared workdir. That roughly halves
wall-clock time versus a single GPU.

**Enable it first:** right sidebar > Session options > Accelerator >
**GPU T4 x2**, and turn Internet **on**.

> **Account warning:** Kaggle has banned accounts that only ever consume GPU
> hours and never take part in competitions. Use this in moderation and take
> part in the site properly. Not a rumour — a reported ban reason.


## 1. Check both GPUs


In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

import torch
n = torch.cuda.device_count()
assert n > 0, 'No GPU. Session options > Accelerator > GPU T4 x2.'
print(f'{n} GPU(s) visible')
if n < 2:
    print('Only one GPU — select "GPU T4 x2" for the parallel path.')


## 2. Install


In [ ]:
!pip install -q -U diffusers transformers accelerate imageio-ffmpeg 2>/dev/null
!git clone -q https://github.com/Naserkhan07/soul_exter.git /kaggle/working/soul_exter 2>/dev/null || true
%cd /kaggle/working/soul_exter
!git checkout -q arena/019f98a2-soul-exter && git pull -q
print('ready')


## 3. Your script

Describe **camera and motion**, not just the subject. Repeat character
details in every scene — the model has no memory between clips.


In [ ]:
script = '''
Scene 1: A lighthouse on a black rock headland at dusk, beam sweeping across
heavy grey water. Rain streaks sideways. Slow dolly in.

Scene 2: Inside the lantern room, brass fittings glowing warm. An old keeper
in a wool coat winds a mechanism by hand.

Scene 3: Waves crash white over a dark reef, spray flung high. Handheld,
violent motion.

Scene 4: A small fishing boat pinned against rocks, mast broken, a lantern
swinging wildly on deck.

Scene 5: The keeper hauls a heavy lever with both hands, straining.

Scene 6: Dawn over a calm flat sea, pale gold light. Two figures wrapped in
blankets on stone steps, steam rising from tin mugs.
'''
open('/kaggle/working/script.txt','w').write(script)
!python -m soulclip.cli scenes /kaggle/working/script.txt --clip-seconds 5


## 4. Settings

Start with `CLIPS = 6`. The run prints a measured per-clip time and an ETA,
so after a few minutes you will know your real rate — then raise to 60.


In [ ]:
CLIPS  = 6            # 60 = a full ~5 minute film
WIDTH, HEIGHT = 512, 320   # 768x512 looks better and costs ~2x
STYLE  = 'cinematic anime, detailed background art, dramatic lighting'

WORKDIR = '/kaggle/working/film/work'
OUTPUT  = '/kaggle/working/film/film.mp4'
print(f'{CLIPS} clips x 5.04s = {CLIPS*5.04:.0f}s of film')


## 5. Generate on both GPUs at once

Each worker takes half the scenes and writes into the same workdir.
`--scenes` workers never stitch, so no partial film is produced.


In [ ]:
import subprocess, os, math

half = math.ceil(CLIPS / 2)
ranges = [(1, half), (half + 1, CLIPS)] if torch.cuda.device_count() > 1 else [(1, CLIPS)]

procs = []
for gpu, (lo, hi) in enumerate(ranges):
    if lo > hi:
        continue
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    cmd = ['python','-m','soulclip.cli','render','/kaggle/working/script.txt',
           '--provider','ltx','--clip-seconds','5',
           '--max-scenes',str(CLIPS),'--target',str(CLIPS*5),
           '--scenes',f'{lo}-{hi}',
           '--width',str(WIDTH),'--height',str(HEIGHT),
           '--style',STYLE,'--workdir',WORKDIR,'-o',OUTPUT]
    print(f'GPU {gpu}: scenes {lo}-{hi}')
    procs.append(subprocess.Popen(cmd, env=env,
                                  stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, text=True))

for gpu, p in enumerate(procs):
    for line in p.stdout:
        print(f'[gpu{gpu}] {line}', end='')
    p.wait()
print('\nboth workers finished')


## 6. Stitch everything

Run without `--scenes`: every clip is reused, nothing regenerates.


In [ ]:
!python -m soulclip.cli render /kaggle/working/script.txt \
    --provider ltx --clip-seconds 5 \
    --max-scenes $CLIPS --target $((CLIPS*5)) \
    --width $WIDTH --height $HEIGHT \
    --workdir "$WORKDIR" -o "$OUTPUT"


## 7. Watch it


In [ ]:
from IPython.display import HTML
from base64 import b64encode
data = b64encode(open(OUTPUT,'rb').read()).decode()
HTML(f'<video width=640 controls src="data:video/mp4;base64,{data}"></video>')


---
## Scaling up

Set `CLIPS = 60` for a ~5m02s film (60 clips x 5.04s at 24fps).

Your real rate comes from the ETA line the run prints:

```
      avg 48s/clip · 52 left · ~42 min to go
```

With two GPUs the wall-clock is roughly half the single-GPU total. Rough
bracket at 512x320 on T4-class hardware: **~25-35 min for 60 clips**, but
trust your measured number over this.

### Free GPU hours, honestly

| Platform | Free allowance | 5-min films/week @512x320 |
|---|---|---|
| Kaggle | 30 h/wk guaranteed | ~29-40 |
| Colab | 15-30 h/wk variable | ~21-29 |
| Both | ~52 h/wk | ~50-69 |

**Nothing is unlimited.** These are large quotas, not infinite ones. Kaggle
is the most generous and most predictable of the free tiers.

### Save your work

`/kaggle/working` persists when you *Save Version*, but is wiped when the
session ends otherwise. Download the film, or commit the notebook, before
you close it.
